# Gold — KPIs de satisfação e qualidade dos dados (`case.gold`)

Lê a fato `ft_chamados` e as dimensões da Gold e grava três tabelas prontas para o painel (Power BI via CSV):

| Tabela | Grão | Uso |
|---|---|---|
| `kpi_csat` | mês x canal x motivo x resolvido x faixa de espera x faixa etária | Cortes do CSAT. Traz numeradores e denominadores para o Power BI recalcular a razão |
| `kpi_csat_mensal` | mês | Evolução do CSAT e teste dos 85% citados pela Marketing |
| `qualidade_dados` | verificação | Inconsistências encontradas, quantas e como foram tratadas |

**Definição do CSAT:** % de chamados com nota 4 ou 5 entre os chamados com nota válida (1 a 5). A nota média fica como indicador de apoio. Cada chamado conta uma vez (a fato usa a tabela de pesquisa por chamado).

**No Power BI, nunca somar percentuais:** use `SUM(qtd_notas_4_5) / SUM(qtd_com_nota)`.

Idempotente (`overwrite`), parametrizado por widgets e com portão de integridade que falha o Job se os totais não fecharem.

In [ ]:
import json
from functools import reduce

from pyspark.sql import functions as F


def parametro(nome, padrao):
    try:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)
    except NameError:
        return padrao


CATALOG = parametro("catalog", "case")
BRONZE_SCHEMA = parametro("bronze_schema", "bronze")
SILVER_SCHEMA = parametro("silver_schema", "silver")
GOLD_SCHEMA = parametro("gold_schema", "gold")
AMOSTRA_MINIMA = int(parametro("amostra_minima", "30"))  # abaixo disso o recorte é sinalizado
FORMATO = "delta"

TABELA_FATO = "ft_chamados"


def fq(schema, tabela):
    return f"`{CATALOG}`.{schema}.{tabela}"


def existe(schema, tabela):
    return spark.catalog.tableExists(fq(schema, tabela))


def gravar(df, tabela):
    (
        df.write.format(FORMATO)
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(fq(GOLD_SCHEMA, tabela))
    )

## 1. Base de cálculo
Junta a fato com as dimensões (categoria e criticidade do motivo, tipo de canal e, se existir, a faixa etária do cliente). As junções são `LEFT`, então nenhum chamado sai da base. O que não casa vira "sem ..." para aparecer no painel em vez de sumir.

In [ ]:
def montar_base():
    ft = spark.table(fq(GOLD_SCHEMA, TABELA_FATO))

    dim_motivo = spark.table(fq(GOLD_SCHEMA, "dim_motivo")).select(
        "id_motivo", F.col("categoria").alias("categoria_motivo"), "criticidade"
    ).dropDuplicates(["id_motivo"])
    dim_canal = spark.table(fq(GOLD_SCHEMA, "dim_canal")).select("id_canal", "tipo_canal").dropDuplicates(["id_canal"])

    dim_cliente = spark.table(fq(GOLD_SCHEMA, "dim_cliente"))
    if "faixa_etaria" in dim_cliente.columns:
        faixa = dim_cliente.select("id_cliente", "faixa_etaria").dropDuplicates(["id_cliente"])
    else:
        faixa = None

    base = ft.join(dim_motivo, "id_motivo", "left").join(dim_canal, "id_canal", "left")
    if faixa is not None:
        base = base.join(faixa, "id_cliente", "left")
    else:
        base = base.withColumn("faixa_etaria", F.lit(None).cast("string"))

    espera = F.col("tempo_espera")  # em segundos, calculado a partir dos horários
    faixa_espera = (
        F.when(espera.isNull(), "sem informacao")
        .when(espera == 0, "sem espera")
        .when(espera <= 300, "ate 5 min")
        .when(espera <= 900, "5 a 15 min")
        .otherwise("mais de 15 min")
    )

    def rotulo(coluna, vazio):
        return F.coalesce(F.col(coluna).cast("string"), F.lit(vazio))

    return base.select(
        "id_chamado",
        rotulo("mes_abertura", "sem data").alias("mes_abertura"),
        rotulo("canal", "sem canal").alias("canal"),
        rotulo("tipo_canal", "sem tipo").alias("tipo_canal"),
        rotulo("motivo", "sem motivo").alias("motivo"),
        rotulo("categoria_motivo", "sem categoria").alias("categoria_motivo"),
        rotulo("criticidade", "sem criticidade").alias("criticidade"),
        rotulo("resolvido", "sem informacao").alias("resolvido"),
        faixa_espera.alias("faixa_espera"),
        rotulo("faixa_etaria", "sem informacao").alias("faixa_etaria"),
        F.col("nota_atendimento").cast("double").alias("nota"),
    )


base = montar_base()
total_chamados = base.count()
print(f"Chamados na base: {total_chamados:,}")

## 2. KPIs de satisfação
`kpi_csat` guarda contagens, não só percentuais: assim qualquer filtro do painel recalcula o CSAT corretamente. `amostra_pequena` marca recortes com poucas respostas, que não devem ser interpretados sozinhos.

In [ ]:
def agregados():
    nota = F.col("nota")
    return [
        F.count(F.lit(1)).alias("qtd_chamados"),
        F.sum(nota.isNotNull().cast("int")).alias("qtd_com_nota"),
        F.sum((nota >= 4).cast("int")).alias("qtd_notas_4_5"),
        F.sum((nota <= 2).cast("int")).alias("qtd_notas_1_2"),
        F.coalesce(F.sum(nota), F.lit(0.0)).alias("soma_notas"),
    ]


def derivados(df):
    return (
        df.withColumn("csat_pct", F.when(F.col("qtd_com_nota") > 0,
                                         F.round(100 * F.col("qtd_notas_4_5") / F.col("qtd_com_nota"), 2)))
        .withColumn("nota_media", F.when(F.col("qtd_com_nota") > 0,
                                         F.round(F.col("soma_notas") / F.col("qtd_com_nota"), 3)))
        .withColumn("taxa_resposta_pct", F.round(100 * F.col("qtd_com_nota") / F.col("qtd_chamados"), 2))
        .withColumn("amostra_pequena", F.col("qtd_com_nota") < AMOSTRA_MINIMA)
    )


CHAVES = ["mes_abertura", "canal", "tipo_canal", "motivo", "categoria_motivo", "criticidade",
          "resolvido", "faixa_espera", "faixa_etaria"]

kpi_csat = derivados(base.groupBy(*CHAVES).agg(*agregados()))
gravar(kpi_csat, "kpi_csat")

kpi_mensal = derivados(base.groupBy("mes_abertura").agg(*agregados())).withColumn(
    "nota_media_pct_escala", F.when(F.col("nota_media").isNotNull(), F.round(F.col("nota_media") / 5 * 100, 2))
).orderBy("mes_abertura")
gravar(kpi_mensal, "kpi_csat_mensal")

print(f"kpi_csat: {spark.table(fq(GOLD_SCHEMA, 'kpi_csat')).count():,} linhas")
print(f"kpi_csat_mensal: {spark.table(fq(GOLD_SCHEMA, 'kpi_csat_mensal')).count():,} linhas")

## 3. Qualidade dos dados
Uma linha por verificação, com quantos registros foram afetados, o total e como o problema foi tratado. É a base do slide de confiabilidade para o Comitê. Verificações cuja tabela de origem não existe na conta são puladas e listadas no aviso final.

In [ ]:
linhas = []
avisos = []


def registrar(camada, tabela, verificacao, afetados, total, tratamento):
    pct = round(100.0 * afetados / total, 2) if total else None
    linhas.append((camada, tabela, verificacao, int(afetados), int(total), pct, tratamento))


def soma(cond):
    return F.coalesce(F.sum(cond.cast("int")), F.lit(0))


# ---- Fato (Gold)
ft = spark.table(fq(GOLD_SCHEMA, TABELA_FATO))
cols = set(ft.columns)
verificacoes_ft = []


def se(coluna, cond, texto, tratamento):
    if coluna in cols:
        verificacoes_ft.append((texto, cond, tratamento))


se("flag_sem_cliente", F.col("flag_sem_cliente"), "Chamado sem cliente na dimensão", "Mantido na fato com flag")
se("flag_sem_custo", F.col("flag_sem_custo"), "Chamado sem custo registrado", "Mantido com custo nulo e flag")
se("nota_atendimento", F.col("nota_atendimento").isNull(), "Chamado sem nota válida de satisfação",
   "Fora do CSAT; contado à parte na taxa de resposta")
se("flag_respondeu", ~F.col("flag_respondeu"), "Chamado sem nenhuma pesquisa respondida",
   "Mantido; entra no denominador da taxa de resposta")
se("flag_sem_atendente", F.col("flag_sem_atendente"), "Chamado sem atendente (id -1)",
   "Mantido com flag; esperado em autosserviço")
se("flag_ordem_invalida", F.col("flag_ordem_invalida"), "Horários fora de ordem cronológica",
   "Tempos ficam nulos; chamado mantido")
se("hora_inicio_atendimento", F.col("hora_inicio_atendimento").isNull(),
   "Chamado sem início de atendimento (indício de abandono)", "Mantido; espera e duração nulas")
se("tempo_espera", F.col("tempo_espera") == 0, "Espera exatamente zero (informativo)",
   "Não é erro; usar mediana e percentis no painel")
if {"tempo_espera", "tempo_espera_original"} <= cols:
    verificacoes_ft.append((
        "Espera calculada pelos horários diverge do campo original",
        F.col("tempo_espera").isNotNull() & F.col("tempo_espera_original").isNotNull()
        & (F.col("tempo_espera") != F.col("tempo_espera_original")),
        "Vale o cálculo pelos horários; original guardado para auditoria"))
if {"tempo_atendimento", "tempo_atendimento_original"} <= cols:
    verificacoes_ft.append((
        "Duração calculada pelos horários diverge do campo original",
        F.col("tempo_atendimento").isNotNull() & F.col("tempo_atendimento_original").isNotNull()
        & (F.col("tempo_atendimento") != F.col("tempo_atendimento_original")),
        "Vale o cálculo pelos horários; original guardado para auditoria"))
se("id_motivo", F.col("id_motivo").isNull(), "Motivo sem correspondência na dimensão",
   "Mantido; aparece como 'sem motivo' no painel")
se("id_canal", F.col("id_canal").isNull(), "Canal sem correspondência na dimensão",
   "Mantido; aparece como 'sem canal' no painel")

total_ft = ft.count()
if verificacoes_ft:
    medidas = ft.agg(*[soma(c).alias(f"v{i}") for i, (_, c, _) in enumerate(verificacoes_ft)]).collect()[0]
    for i, (texto, _, tratamento) in enumerate(verificacoes_ft):
        registrar("gold", TABELA_FATO, texto, medidas[f"v{i}"], total_ft, tratamento)

# ---- Pesquisa de satisfação (Silver)
if existe(SILVER_SCHEMA, "tb_pesquisa_satisfacao"):
    p = spark.table(fq(SILVER_SCHEMA, "tb_pesquisa_satisfacao"))
    pc = set(p.columns)
    total_p = p.count()
    for coluna, texto, trat in [
        ("flag_sem_nota", "Pesquisa sem nota (nula ou texto 'NULL')", "Nota nula, fora do CSAT"),
        ("flag_nota_fora_dominio", "Nota fora da escala 1 a 5 (-1, 0, 6)", "Anulada; valor original guardado"),
        ("flag_chamado_multiplas_pesquisas", "Pesquisa de chamado com mais de uma pesquisa",
         "Mantida na tabela principal; consolidada por chamado"),
    ]:
        if coluna in pc:
            registrar("silver", "tb_pesquisa_satisfacao", texto, p.filter(F.col(coluna)).count(), total_p, trat)
else:
    avisos.append("tb_pesquisa_satisfacao ausente")

if existe(SILVER_SCHEMA, "tb_pesquisa_satisfacao_chamado"):
    pch = spark.table(fq(SILVER_SCHEMA, "tb_pesquisa_satisfacao_chamado"))
    registrar("silver", "tb_pesquisa_satisfacao_chamado", "Chamado com mais de uma pesquisa",
              pch.filter("qtd_pesquisas > 1").count(), pch.count(),
              "Consolidado em 1 linha por chamado; CSAT igual ao por pesquisa")
else:
    avisos.append("tb_pesquisa_satisfacao_chamado ausente")

if existe(SILVER_SCHEMA, "tb_pesquisa_satisfacao_quarentena") and existe(SILVER_SCHEMA, "tb_pesquisa_satisfacao"):
    q = spark.table(fq(SILVER_SCHEMA, "tb_pesquisa_satisfacao_quarentena")).count()
    registrar("silver", "tb_pesquisa_satisfacao", "Pesquisa em quarentena (id nulo ou repetido)",
              q, q + spark.table(fq(SILVER_SCHEMA, "tb_pesquisa_satisfacao")).count(),
              "Separada com o motivo, fora da Silver")

# ---- Clientes e horários (Silver)
if existe(SILVER_SCHEMA, "tb_clientes") and "status_idade" in spark.table(fq(SILVER_SCHEMA, "tb_clientes")).columns:
    cl = spark.table(fq(SILVER_SCHEMA, "tb_clientes"))
    registrar("silver", "tb_clientes", "Idade ausente ou fora de 18 a 120", cl.filter("status_idade <> 'ok'").count(),
              cl.count(), "Idade nula na dimensão; cliente mantido")
else:
    avisos.append("tb_clientes/status_idade ausente")

if existe(SILVER_SCHEMA, "tb_chamados_hora") and "status_integridade_cliente" in spark.table(
        fq(SILVER_SCHEMA, "tb_chamados_hora")).columns:
    ch = spark.table(fq(SILVER_SCHEMA, "tb_chamados_hora"))
    registrar("silver", "tb_chamados_hora", "Cliente do chamado difere entre as duas bases de chamados",
              ch.filter("status_integridade_cliente = 'CLIENTE_DIVERGENTE'").count(), ch.count(),
              "Sinalizado; a fato usa o cliente de tb_chamados")
else:
    avisos.append("tb_chamados_hora/status_integridade_cliente ausente")

# ---- Duplicidade de chamados (Bronze x Silver)
if existe(BRONZE_SCHEMA, "tb_chamados") and existe(SILVER_SCHEMA, "tb_chamados"):
    b = spark.table(fq(BRONZE_SCHEMA, "tb_chamados")).count()
    s = spark.table(fq(SILVER_SCHEMA, "tb_chamados")).count()
    registrar("silver", "tb_chamados", "Chamados duplicados removidos na Silver", b - s, b,
              "Deduplicado por id_chamado")
else:
    avisos.append("tb_chamados (bronze/silver) ausente")

esquema = "camada string, tabela string, verificacao string, qtd_afetados long, qtd_total long, pct double, tratamento string"
qualidade = spark.createDataFrame(linhas, esquema)
gravar(qualidade, "qualidade_dados")
print(f"qualidade_dados: {qualidade.count()} verificações")
if avisos:
    print("Avisos (verificações puladas):", "; ".join(avisos))

## 4. Portão de integridade
Falha o Job se os totais dos KPIs não baterem com a fato ou se a fato tiver `id_chamado` repetido.

In [ ]:
erros = []

ft = spark.table(fq(GOLD_SCHEMA, TABELA_FATO))
n_ft = ft.count()
n_unicos = ft.select("id_chamado").distinct().count()
if n_ft != n_unicos:
    erros.append(f"ft_chamados tem id_chamado repetido ({n_ft} linhas, {n_unicos} ids)")

k = spark.table(fq(GOLD_SCHEMA, "kpi_csat")).agg(
    F.sum("qtd_chamados").alias("c"), F.sum("qtd_com_nota").alias("n"), F.sum("qtd_notas_4_5").alias("p"),
    F.sum("soma_notas").alias("s"),
).collect()[0]
n_notas = ft.filter(F.col("nota_atendimento").isNotNull()).count()
if k["c"] != n_ft:
    erros.append(f"kpi_csat soma {k['c']} chamados; a fato tem {n_ft}")
if k["n"] != n_notas:
    erros.append(f"kpi_csat soma {k['n']} notas; a fato tem {n_notas}")

m = spark.table(fq(GOLD_SCHEMA, "kpi_csat_mensal")).agg(
    F.sum("qtd_chamados").alias("c"), F.sum("qtd_com_nota").alias("n"), F.sum("qtd_notas_4_5").alias("p"),
).collect()[0]
if (m["c"], m["n"], m["p"]) != (k["c"], k["n"], k["p"]):
    erros.append("kpi_csat_mensal não bate com kpi_csat")

if erros:
    raise ValueError("Portão de integridade falhou: " + " | ".join(erros))

resumo = {
    "chamados": int(n_ft),
    "chamados_com_nota": int(n_notas),
    "csat_pct": round(100 * k["p"] / k["n"], 2) if k["n"] else None,
    "nota_media": round(k["s"] / k["n"], 3) if k["n"] else None,
    "taxa_resposta_pct": round(100 * n_notas / n_ft, 2) if n_ft else None,
    "verificacoes_qualidade": int(spark.table(fq(GOLD_SCHEMA, "qualidade_dados")).count()),
}
print(json.dumps(resumo, ensure_ascii=False))
try:
    dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False))
except NameError:
    pass